In [18]:
import random
import numpy as np

In [19]:
random.random()

0.3217460698110688

In [20]:
WIDTH = 4
HEIGHT = 3
WALLS = [(1, 1)]
RATE = 0.8
GAMMA = 0.99
ACTIONS = [[0, 1], [0, -1], [1, 0], [-1, 0]]
INNITIAL_STATE = [0, 0]


"""
width: int
height: int
walls: list of (a, b), index from 0

Return a 2d list of possible squares"""


def create_grid(width, height, walls):
    grid = [[0 for _ in range(width)] for _ in range(height)]
    for wall in walls:
        grid[wall[0]][wall[1]] = 1
    return grid

In [21]:
a = [[0, 0], [1, 0]]
len(a)

2

In [22]:
def is_wall(grid, state):
    row = state[0]
    column = state[1]
    height = len(grid)
    width = len(grid[0])
    if row < 0 or row >= height or column < 0 or column >= width or grid[row][column] == 1:
        return True
    return False

In [23]:
grid = create_grid(WIDTH, HEIGHT, WALLS)
is_wall(grid, (2, 1))

False

In [24]:
def take_action(grid, state, action, rate):
    i = random.random()
    if i < rate:
        # Perform according to action
        height = len(grid)
        width = len(grid[0])
        new_row = state[0] + action[0]
        new_column = state[1] + action[1]
        # if the next state is a wall
        if not is_wall(grid, (new_row, new_column)):
            return new_row, new_column
        return state
        
    elif i >= rate and i < (rate + 1) / 2:
        # Perform the perpendicular to one way
        if action[0] == 0:
            action[0] = 1
            action[1] = 0
        else:
            action[0] = 0
            action[1] = 1
        print("case 2")
        return take_action(grid, state, action, 1)
            
    else:
        # Perform the perpendicular action to another way
        if action[0] == 0:
            action[0] = -1
            action[1] = 0
        else:
            action[0] = 0
            action[1] = -1
        print("case 3")
        return take_action(grid, state, action, 1)

In [58]:
def get_reward(state, grid):
    """Return reward of a state: R(S)"""
    if is_wall(grid, state):
        return None
    if state == [0, 3]:
        return 1
    if state == [1, 3]:
        return -1
    return -0.04

In [144]:
class Agent:
    def __init__(self, training_method):
        self.training_method = training_method

    def train(self, grid):
        if self.training_method == "value":
            # Perform value iteration
            self.value = [0 for _ in range(WIDTH * HEIGHT)]
            for _ in range(1000):
                for s in range(WIDTH * HEIGHT):
                    if not is_wall(grid, num_to_state(s)):
                        self.value[s] = get_reward(num_to_state(s), grid) + GAMMA * max( sum(prob(grid, num_to_state(s), a, RATE)[s_2] * self.value[s_2] for s_2 in range(WIDTH * HEIGHT)) for a in ACTIONS) 
        elif self.training_method == "policy":
            # Perform policy iteration
            self.policy = [random.choice(ACTIONS) for _ in range(WIDTH * HEIGHT)]
            self.training_method = "value"
            for _ in range(1000):
                p_pi = np.array([list(prob(grid, num_to_state(s), self.policy[s], RATE).values()) for s in range(WIDTH * HEIGHT)])
                r = np.array([get_reward(num_to_state(s), grid) for s in range(WIDTH * HEIGHT)])
                r[5] = 0
                self.value = np.linalg.inv((np.identity(WIDTH * HEIGHT) - GAMMA * p_pi)) @ r
                for s in range(WIDTH * HEIGHT):
                    if not is_wall(grid, num_to_state(s)):
                        self.policy[s] = self.pick(num_to_state(s), grid) 

            self.training_method = "policy"

    def pick(self, state, grid):
        """Return the optimal step from a state."""
        if self.training_method == "value":
            a_max = [0, 0]
            expect_max = - float('inf')
            for a in ACTIONS:
                expect = sum(prob(grid, state, a, RATE)[s] * self.value[s] for s in range(WIDTH * HEIGHT))
                if expect > expect_max:
                    expect_max = expect
                    a_max = a
            return a_max
        elif self.training_method == "policy":
            return self.policy[state_to_num(state)]

    def play(self, state, grid, steps):
        """Take actions from state until the end.
        Return the sequence from state to the end state"""
        states = [state]
        for _ in range(steps):
            state = take_action(grid, state, self.pick(state, grid), RATE)
            states.append([state])
        return states

In [145]:
agent = Agent("policy")
agent.train(grid)

In [146]:
agent.policy

[[0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, -1],
 [0, -1],
 [0, 1],
 [0, 1],
 [0, 1],
 [0, -1]]

In [143]:
agent.play([0, 0], grid, 10)

case 3


[[0, 0],
 (0, 1),
 (0, 2),
 (0, 3),
 (0, 2),
 (0, 3),
 (0, 2),
 (0, 3),
 (0, 2),
 (0, 3),
 (0, 2)]

In [113]:
policy = [random.choice(ACTIONS) for _ in range(WIDTH * HEIGHT)]
p_pi = np.array([list(prob(grid, num_to_state(s), policy[s], RATE).values()) for s in range(WIDTH * HEIGHT)])
p_pi, policy

(array([[0.1, 0.1, 0. , 0. , 0.8, 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0.1, 0.8, 0.1, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0.1, 0.8, 0.1, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0. , 0. , 0.1, 0.9, 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
        [0.1, 0. , 0. , 0. , 0.8, 0. , 0. , 0. , 0.1, 0. , 0. , 0. ],
        [0. , 0.1, 0. , 0. , 0. , 0. , 0.8, 0. , 0. , 0.1, 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0.1, 0.1, 0. , 0. , 0.8, 0. ],
        [0. , 0. , 0. , 0.1, 0. , 0. , 0.8, 0. , 0. , 0. , 0. , 0.1],
        [0. , 0. , 0. , 0. , 0.1, 0. , 0. , 0. , 0.1, 0.8, 0. , 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 0.8, 0.1, 0. ],
        [0. , 0. , 0. , 0. , 0. , 0. , 0.8, 0. , 0. , 0.1, 0. , 0.1],
        [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 0.9]]),
 [[1, 0],
  [1, 0],
  [-1, 0],
  [-1, 0],
  [0, -1],
  [0, 1],
  [1, 0],
  [0, -1],
  [0, 1],
  [-1, 0],
  [-1, 0],
  [1, 0]])

In [80]:
r = np.array([get_reward(num_to_state(s), grid) for s in range(WIDTH * HEIGHT)])
r[5] = 0
rel = (np.identity(WIDTH * HEIGHT) - GAMMA * p_pi) @ r
r[5] = 99
rel_2 = (np.identity(WIDTH * HEIGHT) - GAMMA * p_pi) @ r
rel-rel_2

array([[0.0, 0.0, 0.0, 0.0, 0.0, -99.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.00000000000001, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.00000000000001, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.00000000000001, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.00000000000001, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.00000000000001, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.00000000000001, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0],
       [0.0, 0.0, 0.0, 0.0, 0.0, -99.00000000000001, 0.0, 0.0, 0.0, 0.0,
        0.0, 0.0]], 

In [30]:
def state_to_num(state):
    num = state[0] * WIDTH + state[1]
    return num

In [31]:
def num_to_state(num):
    state = [0, 0]
    state[0] = num // WIDTH
    state[1] = num % WIDTH
    return state

In [32]:
# Should've done this before take_action function because take_action can utilize this function
def prob(grid, state, action, rate):
    """Return the probability distribution of new states after performing action at state"""

    p = {i: 0 for i in range(WIDTH * HEIGHT)}
    p[state_to_num(state)] = 0
    new_row = state[0] + action[0]
    new_col = state[1] + action[1]
    new_state = [new_row, new_col]
    if not is_wall(grid, new_state):
        p[state_to_num(new_state)] = rate
    else:
        p[state_to_num(state)] += rate

    if action[0] == 0:
        new_row = state[0] + 1
        new_row_2 = state[0] - 1
        if not is_wall(grid, [new_row, state[1]]):
            p[state_to_num([new_row, state[1]])] = (1 - rate) / 2
        else:
            p[state_to_num(state)] += (1 - rate) / 2
        if not is_wall(grid, [new_row_2, state[1]]):
            p[state_to_num([new_row_2, state[1]])] = (1 - rate) / 2
        else:
            p[state_to_num(state)] += (1 - rate) / 2
            
    elif action[1] == 0:
        new_col = state[1] + 1
        new_col_2 = state[1] - 1
        if not is_wall(grid, [state[0], new_col]):
            p[state_to_num([state[0], new_col])] = (1 - rate) / 2
        else:
            p[state_to_num(state)] += (1 - rate) / 2
        if not is_wall(grid, [state[0], new_col_2]):
            p[state_to_num([state[0], new_col_2])] = (1 - rate) / 2
        else:
            p[state_to_num(state)] += (1 - rate) / 2

    return p

In [33]:
prob(grid, [1, 0], [0, 1], RATE)

{0: 0.09999999999999998,
 1: 0,
 2: 0,
 3: 0,
 4: 0.8,
 5: 0,
 6: 0,
 7: 0,
 8: 0.09999999999999998,
 9: 0,
 10: 0,
 11: 0}

In [34]:
agent.value

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [166]:
initial_state = [0, 0]
for a in ACTIONS:
    expect = sum(prob(grid, initial_state, a, RATE)[s] * agent.value[s] for s in range(WIDTH * HEIGHT))
    print(expect)

84.59938444139038
83.60402227952387
82.96286432758905
83.83781086720978


In [170]:
initial_state = [0, 0]
a_max = [0, 0]
expect_max = - float('inf')
for a in ACTIONS:
    expect = sum(prob(grid, initial_state, a, RATE)[s] * agent.value[s] for s in range(WIDTH * HEIGHT))
    if expect > expect_max:
        expect_max = expect
        a_max = a

print(a_max)
print(expect_max)

[0, 1]
84.59938444139038
